# Step 3b: 3D U-Net spleen segmentation (MONAI)
Task: segment the spleen in abdominal CT (Medical Segmentation Decathlon, Task09_Spleen).
Runtime: **T4 GPU**. Expect about 30-45 minutes for 100 epochs. FOR EDUCATION ONLY. NOT FOR CLINICAL USE.

In [ ]:
!pip -q install --upgrade "monai[nibabel,tqdm]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GITHUB_URL = 'https://github.com/pragya-25/medAI.git'
!git clone {GITHUB_URL} /content/medai 2>/dev/null || (cd /content/medai && git pull)
%cd /content/medai
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 1. Look at the data first
The first call downloads about 1.5 GB to local disk (`/content/msd`).

In [ ]:
from ml.segmentation import make_datasets
train_ds, val_ds = make_datasets()
item = val_ds[0]
print('image', tuple(item['image'].shape), 'label', tuple(item['label'].shape), 'label values', item['label'].unique().tolist())

## 2. Train
Dice loss + 96x96x96 patches + sliding-window validation every 5 epochs. The best checkpoint and `seg_metrics.json` are saved to `MyDrive/medai_seg`.

In [ ]:
from ml.segmentation import train
model, val_ds, result = train(epochs=100)

## 3. Training curves

In [ ]:
import matplotlib.pyplot as plt
h = result['history']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot([r['epoch'] for r in h], [r['train_loss'] for r in h]); ax[0].set_title('Train loss (Dice loss)'); ax[0].set_xlabel('epoch')
v = [r for r in h if 'val_dice' in r]
ax[1].plot([r['epoch'] for r in v], [r['val_dice'] for r in v], marker='o'); ax[1].set_title('Validation Dice'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig('/content/drive/MyDrive/medai_seg/seg_curves.png', dpi=110); plt.show()

## 4. Visual check: ground truth vs prediction

In [ ]:
from ml.segmentation import show_predictions
show_predictions(model, val_ds, '/content/drive/MyDrive/medai_seg/seg_examples.png', n=3)

## 5. Final numbers for the report

In [ ]:
import json
r = json.load(open('/content/drive/MyDrive/medai_seg/seg_metrics.json'))
print('mean val Dice:', round(r['mean_val_dice'], 3), '| min', round(r['min_val_dice'], 3), '| max', round(r['max_val_dice'], 3))
print('per-scan:', [round(x, 3) for x in r['per_scan_dice']])